# Data Checks

Quick probes before writing `aggregate_pitcher_yearly.py`. Each answer feeds the rename maps and the dropped-stats list in `aggregate_table.md`.

FanGraphs and Baseball Reference both sit behind Cloudflare bot protection (HTTP 403), so neither is used. The sources here all answer plain requests:

- **MLB Stats API** — standard pitching stats plus a sabermetrics feed (FIP, xFIP, ERA-, FIP-, WAR)
- **Baseball Savant** — custom leaderboard CSV (expected stats, contact quality, plate discipline, batted-ball mix)
- **pylahman** — Cy Young votes
- **Chadwick register** — bridges Lahman IDs to MLB IDs

In [ ]:
import io

import pandas as pd
import requests
import pylahman
from pybaseball import cache, chadwick_register

cache.enable()
pd.set_option("display.max_columns", None)

YEARS = range(2015, 2026)

## 1. Does Lahman have 2025 Cy Young votes?

In [ ]:
awards = pylahman.AwardsSharePlayers()
cy = awards[(awards["awardID"] == "Cy Young Award") & (awards["yearID"] >= 2015)]

# vote-getters per league-year — 2025 should show up with both AL and NL
cy.groupby(["yearID", "lgID"]).size().unstack()

lgID,AL,NL
yearID,,
2015,14,9
2016,11,10
2017,8,10
2018,9,9
2019,8,11
2020,11,12
2021,12,9
2022,10,12
2023,12,8


## 2. MLB Stats API: standard + sabermetrics

In [ ]:
def mlb_stats(kind, season):
    """One season of pitching stats for every pitcher. kind = 'season' or 'sabermetrics'."""
    params = {
        "stats": kind, "group": "pitching", "season": season,
        "sportId": 1, "playerPool": "ALL", "limit": 5000,
    }
    resp = requests.get("https://statsapi.mlb.com/api/v1/stats", params=params, timeout=60)
    resp.raise_for_status()
    df = pd.json_normalize(resp.json()["stats"][0]["splits"])
    df.columns = [c.removeprefix("stat.") for c in df.columns]
    df["season"] = season
    return df


std = pd.concat([mlb_stats("season", y) for y in YEARS], ignore_index=True)
saber = pd.concat([mlb_stats("sabermetrics", y) for y in YEARS], ignore_index=True)
print(std.shape, saber.shape)
std.groupby("season").size()

(8968, 83) (8968, 34)


season
2015    735
2016    742
2017    755
2018    799
2019    831
2020    735
2021    909
2022    871
2023    863
2024    855
2025    873
dtype: int64

In [ ]:
# every column on offer
print(std.columns.tolist())
print(saber.columns.tolist())

['season', 'numTeams', 'rank', 'age', 'gamesPlayed', 'gamesStarted', 'groundOuts', 'airOuts', 'runs', 'doubles', 'triples', 'homeRuns', 'strikeOuts', 'baseOnBalls', 'intentionalWalks', 'hits', 'hitByPitch', 'avg', 'atBats', 'obp', 'slg', 'ops', 'caughtStealing', 'stolenBases', 'stolenBasePercentage', 'caughtStealingPercentage', 'groundIntoDoublePlay', 'numberOfPitches', 'era', 'inningsPitched', 'wins', 'losses', 'saves', 'saveOpportunities', 'holds', 'blownSaves', 'earnedRuns', 'whip', 'battersFaced', 'outs', 'gamesPitched', 'completeGames', 'shutouts', 'strikes', 'strikePercentage', 'hitBatsmen', 'balks', 'wildPitches', 'pickoffs', 'totalBases', 'groundOutsToAirouts', 'winPercentage', 'pitchesPerInning', 'gamesFinished', 'strikeoutWalkRatio', 'strikeoutsPer9Inn', 'walksPer9Inn', 'hitsPer9Inn', 'runsScoredPer9', 'homeRunsPer9', 'inheritedRunners', 'inheritedRunnersScored', 'catchersInterference', 'sacBunts', 'sacFlies', 'team.id', 'team.name', 'team.link', 'player.id', 'player.fullName

In [ ]:
# raw MLB API names the table would rely on
expected_std = [
    "player.id", "player.fullName", "league.id", "team.name", "numTeams", "age",
    "wins", "losses", "winPercentage", "era", "runs", "earnedRuns",
    "gamesPlayed", "gamesStarted", "inningsPitched", "outs", "numberOfPitches", "strikes",
    "battersFaced", "completeGames", "shutouts", "saves", "saveOpportunities", "blownSaves",
    "holds", "strikeOuts", "strikeoutsPer9Inn", "baseOnBalls", "intentionalWalks",
    "walksPer9Inn", "strikeoutWalkRatio", "hits", "hitsPer9Inn", "avg", "homeRuns",
    "homeRunsPer9", "whip", "obp", "slg", "ops", "hitBatsmen", "doubles", "triples",
    "atBats", "sacFlies", "groundIntoDoublePlay", "wildPitches", "balks",
    "strikePercentage", "pitchesPerInning",
]
expected_saber = ["player.id", "fip", "xfip", "fipMinus", "eraMinus", "war"]

print("missing std:", [c for c in expected_std if c not in std.columns])
print("missing saber:", [c for c in expected_saber if c not in saber.columns])

missing std: []
missing saber: []


In [ ]:
# traded players: one combined row, or one row per team?
dupes = std[std.duplicated(["player.id", "season"], keep=False)]
print(len(dupes), "rows share a player-season")
dupes.sort_values(["season", "player.id"]).head(10)

0 rows share a player-season


,season,numTeams,rank,age,gamesPlayed,gamesStarted,groundOuts,airOuts,runs,doubles,triples,homeRuns,strikeOuts,baseOnBalls,intentionalWalks,hits,hitByPitch,avg,atBats,obp,slg,ops,caughtStealing,stolenBases,stolenBasePercentage,caughtStealingPercentage,groundIntoDoublePlay,numberOfPitches,era,inningsPitched,wins,losses,saves,saveOpportunities,holds,blownSaves,earnedRuns,whip,battersFaced,outs,gamesPitched,completeGames,shutouts,strikes,strikePercentage,hitBatsmen,balks,wildPitches,pickoffs,totalBases,groundOutsToAirouts,winPercentage,pitchesPerInning,gamesFinished,strikeoutWalkRatio,strikeoutsPer9Inn,walksPer9Inn,hitsPer9Inn,runsScoredPer9,homeRunsPer9,inheritedRunners,inheritedRunnersScored,catchersInterference,sacBunts,sacFlies,team.id,team.name,team.link,player.id,player.fullName,player.link,player.firstName,player.lastName,league.id,league.name,league.link,sport.id,sport.link,sport.abbreviation,position.code,position.name,position.type,position.abbreviation


## 3. Savant custom leaderboard

In [ ]:
SAVANT_COLS = [
    "p_quality_start", "k_percent", "bb_percent",
    "xba", "xslg", "xwoba", "xera",
    "exit_velocity_avg", "barrel_batted_rate", "hard_hit_percent",
    "poorlyweak_percent", "solidcontact_percent",
    "z_swing_percent", "oz_swing_percent", "oz_contact_percent", "iz_contact_percent",
    "whiff_percent", "swing_percent", "in_zone_percent", "f_strike_percent",
    "groundballs_percent", "flyballs_percent", "linedrives_percent", "popups_percent",
    "babip",
]


def savant(season):
    url = (
        "https://baseballsavant.mlb.com/leaderboard/custom"
        f"?year={season}&type=pitcher&filter=&min=1&csv=true"
        f"&selections={','.join(SAVANT_COLS)}"
    )
    # Savant rejects Python's default user agent with a 403
    resp = requests.get(url, headers={"User-Agent": "Mozilla/5.0"}, timeout=60)
    resp.raise_for_status()
    return pd.read_csv(io.StringIO(resp.content.decode("utf-8-sig")))


sav = pd.concat([savant(y) for y in YEARS], ignore_index=True)
print(sav.shape)
print("missing:", [c for c in SAVANT_COLS if c not in sav.columns])
sav.groupby("year").size()

(8543, 28)
missing: []


year
2015    725
2016    734
2017    741
2018    772
2019    804
2020    706
2021    845
2022    808
2023    799
2024    800
2025    809
dtype: int64

## 4. Coverage by year

Share of pitchers with a value, per season. A whole year at 0 means the stat gets dropped. Partial gaps are normal for pitchers with only a few innings.

In [ ]:
def coverage(df, year_col, cols):
    cols = [c for c in cols if c in df.columns]
    cov = df.groupby(year_col)[cols].apply(lambda g: g.notna().mean()).T.round(2)
    return cov[(cov < 1).any(axis=1)]


coverage(std, "season", expected_std)

season,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025


In [ ]:
coverage(saber, "season", expected_saber)

season,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025


In [ ]:
coverage(sav, "year", SAVANT_COLS)

year,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025
oz_contact_percent,1.0,1.0,0.99,1.0,0.99,1.0,1.0,1.0,1.0,1.0,1.0


## 5. Does every vote-getter join?

In [ ]:
reg = chadwick_register()
ids = reg[["key_bbref", "key_mlbam", "name_first", "name_last"]]

# Lahman playerID is not always the BBRef ID (montafr01 -> montafr02), so go via People.bbrefID
people = pylahman.People()[["playerID", "bbrefID"]]
joined = cy.merge(people, on="playerID", how="left")
joined = joined.merge(ids, left_on="bbrefID", right_on="key_bbref", how="left")
no_mlbam = joined[joined["key_mlbam"].isna()]
print(len(joined), "vote-getters,", len(no_mlbam), "with no MLB ID")
no_mlbam[["yearID", "lgID", "playerID", "pointsWon"]]

227 vote-getters, 0 with no MLB ID


,yearID,lgID,playerID,pointsWon


In [ ]:
# vote-getters with an MLB ID but no row in a source that season
def has_row(df, id_col, year_col):
    keys = set(zip(df[id_col].astype(int), df[year_col].astype(int)))
    return [
        pd.notna(k) and (int(k), int(y)) in keys
        for k, y in zip(joined["key_mlbam"], joined["yearID"])
    ]


joined["in_std"] = has_row(std, "player.id", "season")
joined["in_saber"] = has_row(saber, "player.id", "season")
joined["in_savant"] = has_row(sav, "player_id", "year")

show = ["yearID", "lgID", "name_first", "name_last", "pointsWon", "in_std", "in_saber", "in_savant"]
joined.loc[~joined[["in_std", "in_saber", "in_savant"]].all(axis=1), show]

,yearID,lgID,name_first,name_last,pointsWon,in_std,in_saber,in_savant


In [ ]:
# league check: Lahman lgID vs the MLB API league (103 = AL, 104 = NL)
api_lg = std.drop_duplicates(["player.id", "season"])[["player.id", "season", "league.id"]]
lg = joined.merge(api_lg, left_on=["key_mlbam", "yearID"], right_on=["player.id", "season"], how="left")
lg["api_lg"] = lg["league.id"].map({103: "AL", 104: "NL"})
lg.loc[lg["api_lg"] != lg["lgID"], ["yearID", "lgID", "api_lg", "name_first", "name_last"]]

,yearID,lgID,api_lg,name_first,name_last
